In [0]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "openpyxl"
])

dbutils.library.restartPython()

In [0]:
target_table = "workspace.census360.bronze_cbsl_telecommunication_services_raw"

table_exists = spark.catalog.tableExists(target_table)

print(f"Table exists: {table_exists}")

if table_exists:
    existing_df = spark.table(target_table)
    print(f"Rows: {existing_df.count()}")
    print(f"Columns: {len(existing_df.columns)}")

In [0]:
target_table = "workspace.census360.bronze_cbsl_telecommunication_services_raw"

bronze_cbsl_telecom_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

created_df = spark.table(target_table)

print(f"Created table: {target_table}")
print(f"Rows: {created_df.count()}")
print(f"Columns: {len(created_df.columns)}")

In [0]:
bronze_table = "workspace.census360.bronze_cbsl_telecommunication_services_raw"

bronze_df = spark.table(bronze_table)

print(f"Rows: {bronze_df.count()}")
print(f"Columns: {len(bronze_df.columns)}")
print("Column names:")
print(bronze_df.columns)

bronze_df.orderBy("source_row_number").show(
    26,
    truncate=False,
    vertical=True
)

In [0]:
import pandas as pd
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)
from pyspark.sql import functions as F

cbsl_excel_path = (
    "/Volumes/workspace/census360/cbsl_raw_files/"
    "Economic and Social Statistics - 2025_chapter_2_e.xlsx"
)

target_table = (
    "workspace.census360."
    "bronze_cbsl_household_income_hies_raw"
)

# Safety check: do not replace an existing table
if spark.catalog.tableExists(target_table):
    raise ValueError(
        f"{target_table} already exists. No data was changed."
    )

# Read the original sheet without interpreting headers
raw_pdf = pd.read_excel(
    cbsl_excel_path,
    sheet_name="Table 2.11",
    header=None,
    engine="openpyxl"
)

# Convert every Excel value to text while preserving nulls
records = []

for row_number, row in raw_pdf.iterrows():
    record = {
        "source_row_number": int(row_number + 1)
    }

    for column_number, value in enumerate(row):
        column_name = f"column_{column_number:02d}"

        record[column_name] = (
            None
            if pd.isna(value)
            else str(value).strip()
        )

    records.append(record)

# Explicit schema prevents mixed Excel types from causing Arrow errors
schema = StructType(
    [
        StructField(
            "source_row_number",
            IntegerType(),
            False
        )
    ]
    + [
        StructField(
            f"column_{column_number:02d}",
            StringType(),
            True
        )
        for column_number in range(raw_pdf.shape[1])
    ]
)

bronze_household_income_df = (
    spark.createDataFrame(records, schema=schema)
    .withColumn(
        "source_file",
        F.lit(
            "Economic and Social Statistics - "
            "2025_chapter_2_e.xlsx"
        )
    )
    .withColumn(
        "source_sheet",
        F.lit("Table 2.11")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

bronze_household_income_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

created_df = spark.table(target_table)

print(f"Created table: {target_table}")
print(f"Rows: {created_df.count()}")
print(f"Columns: {len(created_df.columns)}")

In [0]:
import pandas as pd
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)
from pyspark.sql import functions as F

cbsl_excel_path = (
    "/Volumes/workspace/census360/cbsl_raw_files/"
    "Economic and Social Statistics - 2025_chapter_2_e.xlsx"
)

target_table = (
    "workspace.census360."
    "bronze_cbsl_poverty_indicators_raw"
)

# Safety check: do not replace an existing table
if spark.catalog.tableExists(target_table):
    raise ValueError(
        f"{target_table} already exists. No data was changed."
    )

raw_pdf = pd.read_excel(
    cbsl_excel_path,
    sheet_name="Table 2.13",
    header=None,
    engine="openpyxl"
)

records = []

for row_number, row in raw_pdf.iterrows():
    record = {
        "source_row_number": int(row_number + 1)
    }

    for column_number, value in enumerate(row):
        record[f"column_{column_number:02d}"] = (
            None if pd.isna(value)
            else str(value).strip()
        )

    records.append(record)

schema = StructType(
    [
        StructField(
            "source_row_number",
            IntegerType(),
            False
        )
    ]
    + [
        StructField(
            f"column_{column_number:02d}",
            StringType(),
            True
        )
        for column_number in range(raw_pdf.shape[1])
    ]
)

bronze_poverty_df = (
    spark.createDataFrame(records, schema=schema)
    .withColumn(
        "source_file",
        F.lit(
            "Economic and Social Statistics - "
            "2025_chapter_2_e.xlsx"
        )
    )
    .withColumn(
        "source_sheet",
        F.lit("Table 2.13")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

bronze_poverty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

created_df = spark.table(target_table)

print(f"Created table: {target_table}")
print(f"Rows: {created_df.count()}")
print(f"Columns: {len(created_df.columns)}")

In [0]:
tables_to_check = [
    "workspace.census360.bronze_cbsl_telecommunication_services_raw",
    "workspace.census360.bronze_cbsl_household_income_hies_raw",
    "workspace.census360.bronze_cbsl_poverty_indicators_raw"
]

for table_name in tables_to_check:
    exists = spark.catalog.tableExists(table_name)

    print(f"\nTable: {table_name}")
    print(f"Exists: {exists}")

    if exists:
        table_df = spark.table(table_name)
        print(f"Rows: {table_df.count()}")
        print(f"Columns: {len(table_df.columns)}")

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

cbsl_excel_path = (
    "/Volumes/workspace/census360/cbsl_raw_files/"
    "Economic and Social Statistics - 2025_chapter_2_e.xlsx"
)

target_table = (
    "workspace.census360."
    "bronze_cbsl_poverty_indicators_raw"
)

if spark.catalog.tableExists(target_table):
    raise ValueError(
        f"{target_table} already exists. No data was changed."
    )

raw_pdf = pd.read_excel(
    cbsl_excel_path,
    sheet_name="Table 2.13",
    header=None,
    engine="openpyxl"
)

records = []

for row_number, row in raw_pdf.iterrows():
    record = {
        "source_row_number": int(row_number + 1)
    }

    for column_number, value in enumerate(row):
        record[f"column_{column_number:02d}"] = (
            None
            if pd.isna(value)
            else str(value).strip()
        )

    records.append(record)

schema = StructType(
    [
        StructField(
            "source_row_number",
            IntegerType(),
            False
        )
    ]
    + [
        StructField(
            f"column_{column_number:02d}",
            StringType(),
            True
        )
        for column_number in range(raw_pdf.shape[1])
    ]
)

bronze_poverty_df = (
    spark.createDataFrame(records, schema=schema)
    .withColumn(
        "source_file",
        F.lit(
            "Economic and Social Statistics - "
            "2025_chapter_2_e.xlsx"
        )
    )
    .withColumn(
        "source_sheet",
        F.lit("Table 2.13")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

bronze_poverty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

created_df = spark.table(target_table)

print(f"Created table: {target_table}")
print(f"Rows: {created_df.count()}")
print(f"Columns: {len(created_df.columns)}")

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

cbsl_excel_path = (
    "/Volumes/workspace/census360/cbsl_raw_files/"
    "Economic and Social Statistics - 2025_chapter_2_e.xlsx"
)

target_table = (
    "workspace.census360."
    "bronze_cbsl_provincial_socioeconomic_raw"
)

# Safety check: do not overwrite an existing table
if spark.catalog.tableExists(target_table):
    raise ValueError(
        f"{target_table} already exists. No data was changed."
    )

raw_pdf = pd.read_excel(
    cbsl_excel_path,
    sheet_name="Table 2.18",
    header=None,
    engine="openpyxl"
)

records = []

for row_number, row in raw_pdf.iterrows():
    record = {
        "source_row_number": int(row_number + 1)
    }

    for column_number, value in enumerate(row):
        record[f"column_{column_number:02d}"] = (
            None
            if pd.isna(value)
            else str(value).strip()
        )

    records.append(record)

schema = StructType(
    [
        StructField(
            "source_row_number",
            IntegerType(),
            False
        )
    ]
    + [
        StructField(
            f"column_{column_number:02d}",
            StringType(),
            True
        )
        for column_number in range(raw_pdf.shape[1])
    ]
)

bronze_provincial_socioeconomic_df = (
    spark.createDataFrame(records, schema=schema)
    .withColumn(
        "source_file",
        F.lit(
            "Economic and Social Statistics - "
            "2025_chapter_2_e.xlsx"
        )
    )
    .withColumn(
        "source_sheet",
        F.lit("Table 2.18")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

bronze_provincial_socioeconomic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

created_df = spark.table(target_table)

print(f"Created table: {target_table}")
print(f"Rows: {created_df.count()}")
print(f"Columns: {len(created_df.columns)}")

In [0]:
tables_to_check = [
    "workspace.census360.bronze_cbsl_telecommunication_services_raw",
    "workspace.census360.bronze_cbsl_household_income_hies_raw",
    "workspace.census360.bronze_cbsl_poverty_indicators_raw"
]

for table_name in tables_to_check:
    exists = spark.catalog.tableExists(table_name)

    print(f"\nTable: {table_name}")
    print(f"Exists: {exists}")

    if exists:
        table_df = spark.table(table_name)
        print(f"Rows: {table_df.count()}")
        print(f"Columns: {len(table_df.columns)}")

In [0]:
target_table = (
    "workspace.census360."
    "bronze_cbsl_provincial_socioeconomic_raw"
)

exists = spark.catalog.tableExists(target_table)

print(f"Table: {target_table}")
print(f"Exists: {exists}")

if exists:
    table_df = spark.table(target_table)
    print(f"Rows: {table_df.count()}")
    print(f"Columns: {len(table_df.columns)}")